In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical

In [18]:
# Load the dataset
df = pd.read_csv("Darknet.csv")

# Drop unnamed columns
df.drop(columns=[col for col in df.columns if "Unnamed" in col], inplace=True)

# Replace inf/-inf with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Select numeric data and drop NaNs
numeric_df = df.select_dtypes(include=np.number)
df = df.loc[numeric_df.dropna().index]  # drop rows with NaNs in numeric columns
numeric_df = numeric_df.dropna()

# Standardize numeric features
scaler = StandardScaler()
X = scaler.fit_transform(numeric_df)

# Encode 'Label.1' for classification
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['Label.1'])

In [19]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

In [20]:
# Build model
model = Sequential([
    Input(shape=(X.shape[1],)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(y_train_cat.shape[1], activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Train
history = model.fit(X_train, y_train_cat, epochs=20, batch_size=32, validation_data=(X_test, y_test_cat))

Epoch 1/20
3717/3717 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.7335 - loss: 0.8546 - val_accuracy: 0.7990 - val_loss: 0.6029
Epoch 2/20
3717/3717 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.8083 - loss: 0.5657 - val_accuracy: 0.8131 - val_loss: 0.5482
Epoch 3/20
3717/3717 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - accuracy: 0.8170 - loss: 0.5241 - val_accuracy: 0.8098 - val_loss: 0.5347
Epoch 4/20
3717/3717 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - accuracy: 0.8235 - loss: 0.5051 - val_accuracy: 0.8147 - val_loss: 0.5314
Epoch 5/20
3717/3717 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - accuracy: 0.8259 - loss: 0.4867 - val_accuracy: 0.8228 - val_loss: 0.5065
Epoch 6/20
3717/3717 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.8296 - loss: 0.4768 - val_accuracy: 0.8272 - val_loss: 0.4895
Epoch 7/20
3717/3717 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.8326 - loss: 0.4670 - val_accuracy: 0.8269 - val_loss: 0.4830
Epoch 8/20
3717/3717 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.8351 - loss: 0

In [ ]:
# Evaluate
loss, accuracy = model.evaluate(X_test, y_test_cat)
print(f"\n✅ Test Accuracy: {accuracy:.4f}")

# Predictions
y_pred = model.predict(X_test)
y_pred = np.argmax(y_pred, axis=1)
y_true = y_test

In [ ]:
# Classification Report
print("\n📊 Classification Report:")
labels = sorted(np.unique(y_true))
target_names = label_encoder.inverse_transform(labels)
print(classification_report(y_true, y_pred, labels=labels, target_names=target_names))

In [ ]:
# Confusion Matrix
plt.figure(figsize=(12, 8))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()